# Resume Matching Engine using TF-IDF and Cosine Similarity

This notebook implements a resume matching pipeline using only standard Python libraries.

The workflow includes:
- skill normalization and alias mapping
- deduplication of canonical skills
- vocabulary construction
- TF-IDF vector generation
- binary vector construction for job descriptions
- cosine similarity based ranking

The implementation follows the exact constraints and formulas specified in the problem statement.

## Skill Normalization and Deduplication

In this stage:
- raw skill strings are cleaned and normalized
- aliases are mapped to canonical skill names
- duplicate skills are removed while preserving order
- unknown skills are discarded

In [1]:
import math
import re

SKILL_ALIASES = {
    "python": "python", "pyhton": "python", "java": "java",
    "javascript": "javascript", "javascrpit": "javascript", "js": "javascript",
    "typescript": "typescript", "typescrpit": "typescript",
    "c++": "cpp", "cpp": "cpp", "r": "r", "kotlin": "kotlin",
    "machinelearning": "machine_learning", "machine learning": "machine_learning",
    "ml": "machine_learning", "sklearn": "machine_learning",
    "deeplearning": "deep_learning", "deep learning": "deep_learning",
    "deep-learning": "deep_learning",
    "tensorflow": "tensorflow", "pytorch": "pytorch", "keras": "keras",
    "nlp": "nlp", "bert": "bert", "xgboost": "xgboost",
    "feature engineering": "feature_engineering",
    "statistics": "statistics", "stats": "statistics",
    "regression": "regression", "clustering": "clustering",
    "data-viz": "data_visualization", "data visualization": "data_visualization",
    "data viz": "data_visualization", "matplotlib": "data_visualization",
    "tableau": "data_visualization", "power-bi": "data_visualization",
    "power bi": "data_visualization", "powerbi": "data_visualization",
    "pandas": "pandas", "numpy": "numpy",
    "react": "react", "reacts": "react", "reactjs": "react",
    "vue": "vue", "vue.js": "vue", "vuejs": "vue",
    "redux": "redux", "tailwind": "tailwind",
    "html/css": "html_css", "html css": "html_css",
    "html": "html_css", "css": "html_css",
    "jest": "jest", "graphql": "graphql",
    "node.js": "nodejs", "nodejs": "nodejs", "node js": "nodejs",
    "flask": "flask",
    "spring boot": "spring_boot", "springboot": "spring_boot",
    "rest api": "rest_api", "rest": "rest_api", "restapi": "rest_api",
    "microservices": "microservices",
    "sql": "sql", "mysql": "mysql", "mysq": "mysql",
    "postgresql": "postgresql", "postgres": "postgresql",
    "mongodb": "mongodb", "redis": "redis", "docker": "docker",
    "kubernetes": "kubernetes", "kubernates": "kubernetes", "k8s": "kubernetes",
    "ci/cd": "ci_cd", "cicd": "ci_cd", "ci cd": "ci_cd",
    "aws": "aws", "android": "android", "firebase": "firebase",
    "algorithms": "algorithms", "algoritms": "algorithms",
    "data structure": "data_structures", "data structures": "data_structures",
    "competitive programming": "competitive_programming",
    "ui/ux": "ui_ux", "ui ux": "ui_ux", "figma": "figma",
}

RESUMES = {
    "Arjun Sharma":    "Pyhton, MachineLearning, SQL, pandas, numpy, Deep-learning",
    "Priya Nair":      "JavaScrpit, Reacts, Node.JS, MongoDb, REST api, HTML/CSS",
    "Rahul Gupta":     "Java, Spring Boot, MySql, Microservices, Docker, kubernates",
    "Sneha Patel":     "Python, TensorFlow, Keras, NLP, BERT, data-viz, matplotlib",
    "Vikram Singh":    "C++, Algoritms, Data Structure, competitive programming, python",
    "Ananya Krishnan": "javascript, vue.js, python, flask, PostgreSQL, AWS, CI/CD",
    "Karan Mehta":     "Python, Sklearn, XGboost, feature engineering, SQL, tableau",
    "Deepika Rao":     "Java, Android, Kotlin, Firebase, REST, UI/UX, figma",
    "Aditya Kumar":    "Reactjs, TypeScrpit, GraphQL, redux, tailwind, nodejs, jest",
    "Meera Iyer":      "python, R, statistics, ML, regression, clustering, Power-BI",
}

SORTED_KEYS = sorted(SKILL_ALIASES.keys(), key=len, reverse=True)

def normalize_and_deduplicate(raw_str):
    tokens = raw_str.split(",")
    seen = set()
    result = []
    for token in tokens:
        cleaned = re.sub(r'\s+', ' ', token.strip()).lower()
        for alias_key in SORTED_KEYS:
            if cleaned == alias_key:
                canonical = SKILL_ALIASES[alias_key]
                if canonical not in seen:
                    seen.add(canonical)
                    result.append(canonical)
                break
    return result

normalized = {}
for name, raw in RESUMES.items():
    skills = normalize_and_deduplicate(raw)
    normalized[name] = skills
    print(f"Candidate: {name}")
    print(f"Normalized: {skills}")
    print(f"N = {len(skills)}")
    print()

Candidate: Arjun Sharma
Normalized: ['python', 'machine_learning', 'sql', 'pandas', 'numpy', 'deep_learning']
N = 6

Candidate: Priya Nair
Normalized: ['javascript', 'react', 'nodejs', 'mongodb', 'rest_api', 'html_css']
N = 6

Candidate: Rahul Gupta
Normalized: ['java', 'spring_boot', 'mysql', 'microservices', 'docker', 'kubernetes']
N = 6

Candidate: Sneha Patel
Normalized: ['python', 'tensorflow', 'keras', 'nlp', 'bert', 'data_visualization']
N = 6

Candidate: Vikram Singh
Normalized: ['cpp', 'algorithms', 'data_structures', 'competitive_programming', 'python']
N = 5

Candidate: Ananya Krishnan
Normalized: ['javascript', 'vue', 'python', 'flask', 'postgresql', 'aws', 'ci_cd']
N = 7

Candidate: Karan Mehta
Normalized: ['python', 'machine_learning', 'xgboost', 'feature_engineering', 'sql', 'data_visualization']
N = 6

Candidate: Deepika Rao
Normalized: ['java', 'android', 'kotlin', 'firebase', 'rest_api', 'ui_ux', 'figma']
N = 7

Candidate: Aditya Kumar
Normalized: ['react', 'typescrip

## Vocabulary Construction

A shared vocabulary is created using normalized resume skills only.

The vocabulary is:
- unique
- alphabetically sorted
- used consistently across resume and JD vectors

In [2]:
#constructing the vocabulary for the same
all_skills = []
for skills in normalized.values():
    all_skills.extend(skills)

vocabulary = sorted(list(set(all_skills)))

print(f"Total vocabulary size: {len(vocabulary)}")
print()
for i, skill in enumerate(vocabulary):
    print(f"{i}: {skill}")

Total vocabulary size: 48

0: algorithms
1: android
2: aws
3: bert
4: ci_cd
5: clustering
6: competitive_programming
7: cpp
8: data_structures
9: data_visualization
10: deep_learning
11: docker
12: feature_engineering
13: figma
14: firebase
15: flask
16: graphql
17: html_css
18: java
19: javascript
20: jest
21: keras
22: kotlin
23: kubernetes
24: machine_learning
25: microservices
26: mongodb
27: mysql
28: nlp
29: nodejs
30: numpy
31: pandas
32: postgresql
33: python
34: r
35: react
36: redux
37: regression
38: rest_api
39: spring_boot
40: sql
41: statistics
42: tailwind
43: tensorflow
44: typescript
45: ui_ux
46: vue
47: xgboost


In [3]:
'''computing document frequencies for all skills in the vocabulary'''

document_frequency = {}

for skill in vocabulary:

    count = 0

    for skills in normalized.values():

        if skill in skills:
            count += 1

    document_frequency[skill] = count

print("Document Frequencies")

for skill, df in document_frequency.items():
    print(f"{skill}: {df}")

Document Frequencies
algorithms: 1
android: 1
aws: 1
bert: 1
ci_cd: 1
clustering: 1
competitive_programming: 1
cpp: 1
data_structures: 1
data_visualization: 3
deep_learning: 1
docker: 1
feature_engineering: 1
figma: 1
firebase: 1
flask: 1
graphql: 1
html_css: 1
java: 2
javascript: 2
jest: 1
keras: 1
kotlin: 1
kubernetes: 1
machine_learning: 3
microservices: 1
mongodb: 1
mysql: 1
nlp: 1
nodejs: 2
numpy: 1
pandas: 1
postgresql: 1
python: 6
r: 1
react: 2
redux: 1
regression: 1
rest_api: 2
spring_boot: 1
sql: 2
statistics: 1
tailwind: 1
tensorflow: 1
typescript: 1
ui_ux: 1
vue: 1
xgboost: 1


## TF-IDF Vector Construction

TF-IDF vectors are generated for resumes using:

TF = 1 / N  
IDF = ln(10 / df)

where:
- N = total unique skills in a resume
- df = number of resumes containing the skill

In [4]:
'''Identifying the TF IDF vectors'''

tfidf_vectors = {}

for name, skills in normalized.items():
    N = len(skills)
    vector = {}
    for skill in vocabulary:
        if skill in skills:
            tf = 1 / N
            idf = math.log(10 / document_frequency[skill])
            vector[skill] = tf * idf
        else:
            vector[skill] = 0.0
    tfidf_vectors[name] = vector

for name, vector in tfidf_vectors.items():
    print(f"\n{name} (N={len(normalized[name])}):")
    for skill, val in vector.items():
        if val > 0:
            print(f"  {skill}: {val:.6f}")


Arjun Sharma (N=6):
  deep_learning: 0.383764
  machine_learning: 0.200662
  numpy: 0.383764
  pandas: 0.383764
  python: 0.085138
  sql: 0.268240

Priya Nair (N=6):
  html_css: 0.383764
  javascript: 0.268240
  mongodb: 0.383764
  nodejs: 0.268240
  react: 0.268240
  rest_api: 0.268240

Rahul Gupta (N=6):
  docker: 0.383764
  java: 0.268240
  kubernetes: 0.383764
  microservices: 0.383764
  mysql: 0.383764
  spring_boot: 0.383764

Sneha Patel (N=6):
  bert: 0.383764
  data_visualization: 0.200662
  keras: 0.383764
  nlp: 0.383764
  python: 0.085138
  tensorflow: 0.383764

Vikram Singh (N=5):
  algorithms: 0.460517
  competitive_programming: 0.460517
  cpp: 0.460517
  data_structures: 0.460517
  python: 0.102165

Ananya Krishnan (N=7):
  aws: 0.328941
  ci_cd: 0.328941
  flask: 0.328941
  javascript: 0.229920
  postgresql: 0.328941
  python: 0.072975
  vue: 0.328941

Karan Mehta (N=6):
  data_visualization: 0.200662
  feature_engineering: 0.383764
  machine_learning: 0.200662
  python

## Job Description Vector Construction

Job descriptions are normalized using the same preprocessing pipeline.

Binary vectors are then created over the shared vocabulary space:
- 1 indicates skill presence
- 0 indicates skill absence

In [5]:
'''
JOB DESCRIPTIONS ARE AS FOLLOWS:
'''
JDS = {
    "JD-1 | Kakao | ML Engineer":
        "Python, Machine Learning, Deep Learning, TensorFlow, PyTorch, SQL, Data Visualization, NLP, BERT, Feature Engineering, Statistics",

    "JD-2 | Naver | Backend Engineer":
        "Java, Spring Boot, MySQL, PostgreSQL, Microservices, Docker, Kubernetes, REST API, CI/CD, Redis",

    "JD-3 | Line | Frontend Engineer":
        "JavaScript, React, Vue, TypeScript, REST API, HTML/CSS, Node.js, GraphQL, Redux, Jest, AWS"
}

'''
NORMALIZING JD SKILLS
'''
normalized_jds = {}

for jd_name, raw_skills in JDS.items():

    skills = normalize_and_deduplicate(raw_skills)

    normalized_jds[jd_name] = skills

    print(f"{jd_name}")
    print(f"Normalized Skills: {skills}")
    print(f"N = {len(skills)}")
    print()


''' BUILDing BINARY JD VECTORS'''


jd_vectors = {}

for jd_name, skills in normalized_jds.items():

    vector = {}

    for skill in vocabulary:

        if skill in skills:
            vector[skill] = 1
        else:
            vector[skill] = 0

    jd_vectors[jd_name] = vector

''' finalising the non_zero entries'''

for jd_name, vector in jd_vectors.items():

    print(f"\n{jd_name}")

    for skill, value in vector.items():

        if value == 1:
            print(f"  {skill}: {value}")

JD-1 | Kakao | ML Engineer
Normalized Skills: ['python', 'machine_learning', 'deep_learning', 'tensorflow', 'pytorch', 'sql', 'data_visualization', 'nlp', 'bert', 'feature_engineering', 'statistics']
N = 11

JD-2 | Naver | Backend Engineer
Normalized Skills: ['java', 'spring_boot', 'mysql', 'postgresql', 'microservices', 'docker', 'kubernetes', 'rest_api', 'ci_cd', 'redis']
N = 10

JD-3 | Line | Frontend Engineer
Normalized Skills: ['javascript', 'react', 'vue', 'typescript', 'rest_api', 'html_css', 'nodejs', 'graphql', 'redux', 'jest', 'aws']
N = 11


JD-1 | Kakao | ML Engineer
  bert: 1
  data_visualization: 1
  deep_learning: 1
  feature_engineering: 1
  machine_learning: 1
  nlp: 1
  python: 1
  sql: 1
  statistics: 1
  tensorflow: 1

JD-2 | Naver | Backend Engineer
  ci_cd: 1
  docker: 1
  java: 1
  kubernetes: 1
  microservices: 1
  mysql: 1
  postgresql: 1
  rest_api: 1
  spring_boot: 1

JD-3 | Line | Frontend Engineer
  aws: 1
  graphql: 1
  html_css: 1
  javascript: 1
  jest: 

## Cosine Similarity and Ranking

Cosine similarity is computed between:
- resume TF-IDF vectors
- JD binary vectors

Candidates are ranked in descending similarity order.
Alphabetical ordering is used for tie-breaking.

In [6]:
'''computing cosine similarity between resume tf-idf vectors and jd binary vectors'''

def cosine_similarity(resume_vector, jd_vector):
    dot_product = 0.0
    resume_norm = 0.0
    jd_norm = 0.0

    for skill in vocabulary:

        a = resume_vector[skill]
        b = jd_vector[skill]

        dot_product += a * b
        resume_norm += a * a
        jd_norm += b * b

    resume_norm = math.sqrt(resume_norm)
    jd_norm = math.sqrt(jd_norm)

    if resume_norm == 0 or jd_norm == 0:
        return 0.0

    return dot_product / (resume_norm * jd_norm)

'''calculating similarity scores for every resume against every jd'''

results = {}

for jd_name, jd_vector in jd_vectors.items():

    scores = []

    for candidate_name, resume_vector in tfidf_vectors.items():
        similarity = cosine_similarity(resume_vector, jd_vector)
        scores.append((candidate_name, similarity))

    scores.sort(key=lambda x: (-x[1], x[0]))
    results[jd_name] = scores

'''printing final top 3 ranked candidates for each jd'''

for jd_name, scores in results.items():

    parts = jd_name.split(" | ")

    jd_id = parts[0]
    company = parts[1]
    role = parts[2]

    print(f"{jd_id} — {company} ({role})")

    top_3 = scores[:3]

    formatted = []

    for candidate_name, score in top_3:
        formatted.append(f"{candidate_name}({score:.2f})")

    print(", ".join(formatted))
    print()

JD-1 — Kakao (ML Engineer)
Sneha Patel(0.57), Karan Mehta(0.53), Arjun Sharma(0.40)

JD-2 — Naver (Backend Engineer)
Rahul Gupta(0.81), Ananya Krishnan(0.28), Deepika Rao(0.19)

JD-3 — Line (Frontend Engineer)
Aditya Kumar(0.67), Priya Nair(0.58), Ananya Krishnan(0.35)

